With the assistance of [From Mr Carrot to Mr Trček: Correcting proper names in machine translation](https://medium.com/@taja.kuzman/from-mr-carrot-to-mr-tr%C4%8Dek-correcting-proper-names-in-machine-translation-49987351c524) and chatgpt4-o

## 範例一
source_sentence, target_sentence 和 word_alignment 是定好的

In [ ]:
# 透過新增邊界檢查 [adding boundary checks] 來調整程式碼以處理索引問題 [Adjust the code to handle index issues]

# 修正後的來源語句、目標語句和對齊語句範例
source_sentence = ["Gospod", "Trček", "je", "obiskal", "Ljubljano"]
target_sentence = ["Mr.", "Carrot", "visited", "Ljubljana"]
word_alignment = {0: 0, 1: 1, 2: 2, 3: 3, 4: 3}  # Adjust alignment for "Ljubljano" to "Ljubljana"

# 命名實體 (不應翻譯的專有名詞)
named_entities = {"Trček", "Ljubljano"}

def correct_translation(source_sentence, target_sentence, word_alignment, named_entities):
    corrected_sentence = target_sentence.copy()

    for source_index, target_index in word_alignment.items():
        # 確保目標索引在目標句的範圍內 [Ensure target index is within the bounds of the target sentence]
        if target_index < len(target_sentence):
            source_word = source_sentence[source_index]
            if source_word in named_entities:
                # 如果目標單字是命名實體，則將其替換為來源單字 [Replace the target word with the source word if it's a named entity]
                corrected_sentence[target_index] = source_word

    return corrected_sentence

# Correct the translation
corrected_translation = correct_translation(source_sentence, target_sentence, word_alignment, named_entities)

# Output the corrected translation
corrected_translation #輸出: ['Mr.', 'Trček', 'visited', 'Ljubljano']



['Mr.', 'Trček', 'visited', 'Ljubljano']

### 關於 word_alignment

`word_alignment` 是一個用來描述源語言（斯洛維尼亞文）和目標語言（英文）之間詞對齊的對應關係的數據結構。具體來說，它是一個字典，鍵是斯洛維尼亞文中的詞的索引，值是英文中對應詞的索引。

讓我們以範例中的句子來進行詳細說明：

### 斯洛維尼亞文：
`["Janez", "Novak", "je", "obiskal", "Maribor"]`

### 錯誤翻譯的英文：
`["John", "Carrot", "visited", "Maribor"]`

### `word_alignment` 對應關係：
```python
word_alignment = {0: 0, 1: 1, 3: 2, 4: 3}
```

### 對應關係逐步解釋：
1. **索引 0**（斯洛維尼亞文 `Janez`）對應到英文的索引 0（`John`）：
   - `Janez` 在斯洛維尼亞文中的位置是第 0 個詞，錯誤地翻譯為英文的第 0 個詞 `John`。
   - 在修正過程中，因為 `Janez` 是專有名詞，我們會將英文中的 `John` 替換回 `Janez`。

2. **索引 1**（斯洛維尼亞文 `Novak`）對應到英文的索引 1（`Carrot`）：
   - `Novak` 在斯洛維尼亞文中的位置是第 1 個詞，錯誤地翻譯為英文的第 1 個詞 `Carrot`。
   - 在修正過程中，`Novak` 也是專有名詞，我們會將英文中的 `Carrot` 替換回 `Novak`。

3. **索引 3**（斯洛維尼亞文 `obiskal`）對應到英文的索引 2（`visited`）：
   - `obiskal` 在斯洛維尼亞文中的位置是第 3 個詞，正確地翻譯為英文的第 2 個詞 `visited`。
   - 這個詞不是專有名詞，因此不需要修改。

4. **索引 4**（斯洛維尼亞文 `Maribor`）對應到英文的索引 3（`Maribor`）：
   - `Maribor` 在斯洛維尼亞文中的位置是第 4 個詞，正確地翻譯為英文的第 3 個詞 `Maribor`。
   - 這裡的翻譯是正確的，因為它是專有名詞且沒有被改動，所以我們不需要進行修正。

### 總結：
- `word_alignment` 表示的是斯洛維尼亞文的每個詞和英文翻譯中對應詞的位置對應。
- 我們根據這些對應關係來決定哪個詞在翻譯過程中可能被錯誤處理，特別是專有名詞部分。然後我們根據這些對應關係將錯誤翻譯的專有名詞替換回來。

這種對應是通過機器翻譯中的詞對齊技術（如 IBM Model 1 或更複雜的對齊模型）來生成的，可以自動計算出不同語言之間的詞對應。

## 範例二

### 步驟 1：取得斯洛維尼亞語的原始文本

In [ ]:
# 斯洛維尼亞語例句
sentence = "Ljubljana je glavno mesto Slovenije."

### 步驟 2：使用 CLASSLA 進行語言處理

CLASSLA 是專門用於斯洛維尼亞語的語言處理工具包，我們將它用來標註和進行詞性標記。

In [ ]:
!pip install -qU classla

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.9/249.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.0/17.0 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.8/407.8 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 776.3/776.3 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.2/76.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 8.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
albucore 0.0.14 requires numpy>=1.24, but you have numpy 1.23.0 which is incompatible.
albumentations 1.4.14 

In [ ]:
import classla

# 初始化 CLASSLA pipeline
classla.download('sl')
nlp = classla.Pipeline('sl')


# 使用 CLASSLA 進行處理
doc = nlp(sentence)
print(doc)

/usr/local/lib/python3.10/dist-packages/requests/__init__.py:109: RequestsDependencyWarning: urllib3 (1.26.20) or chardet (5.2.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  warnings.warn(
INFO:classla:Downloading these customized packages for language: sl (Slovenian)...
| Processor | Package  |
------------------------
| tokenize  | standard |
| pos       | standard |
| lemma     | standard |
| depparse  | standard |
| ner       | standard |
| pretrain  | standard |

INFO:classla:Finished downloading models and saved to /root/classla_resources.
INFO:classla:Loading these models for language: sl (Slovenian):
| Processor | Package  |
------------------------
| tokenize  | standard |
| pos       | standard |
| lemma     | standard |
| depparse  | standard |
| ner       | standard |

INFO:classla:Use device: cpu
INFO:classla:Loading: tokenize
INFO:classla:Loading: pos
INFO:classla:Loading: lemma
INFO:classla:Loading: depparse
INFO:classla:Loading: ner
INFO:classla:Don

[
  [
    [
      {
        "id": 1,
        "text": "Ljubljana",
        "lemma": "Ljubljana",
        "upos": "PROPN",
        "xpos": "Npfsn",
        "feats": "Case=Nom|Gender=Fem|Number=Sing",
        "head": 4,
        "deprel": "nsubj",
        "ner": "B-LOC"
      },
      {
        "id": 2,
        "text": "je",
        "lemma": "biti",
        "upos": "AUX",
        "xpos": "Va-r3s-n",
        "feats": "Mood=Ind|Number=Sing|Person=3|Polarity=Pos|Tense=Pres|VerbForm=Fin",
        "head": 4,
        "deprel": "cop",
        "ner": "O"
      },
      {
        "id": 3,
        "text": "glavno",
        "lemma": "glaven",
        "upos": "ADJ",
        "xpos": "Agpnsn",
        "feats": "Case=Nom|Degree=Pos|Gender=Neut|Number=Sing",
        "head": 4,
        "deprel": "amod",
        "ner": "O"
      },
      {
        "id": 4,
        "text": "mesto",
        "lemma": "mesto",
        "upos": "NOUN",
        "xpos": "Ncnsn",
        "feats": "Case=Nom|Gender=Neut|Number=Sing",


### 步驟 3：使用機器翻譯系統翻譯斯洛維尼亞語文本
這裡使用 https://medium.com/@taja.kuzman/use-google-translate-opus-mt-and-facebook-mt-models-with-just-a-few-lines-in-python-45ada098c4e9 的 google翻譯

---



In [ ]:
!pip install -qU googletrans-py

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.1 MB/s eta 0:00:00


In [ ]:
import googletrans
from googletrans import Translator

# See a list of available languages and language codes
print(googletrans.LANGUAGES)

{'af': 'afrikaans', 'sq': 'albanian', 'am': 'amharic', 'ar': 'arabic', 'hy': 'armenian', 'az': 'azerbaijani', 'eu': 'basque', 'be': 'belarusian', 'bn': 'bengali', 'bs': 'bosnian', 'bg': 'bulgarian', 'ca': 'catalan', 'ceb': 'cebuano', 'ny': 'chichewa', 'zh-cn': 'chinese (simplified)', 'zh-tw': 'chinese (traditional)', 'co': 'corsican', 'hr': 'croatian', 'cs': 'czech', 'da': 'danish', 'nl': 'dutch', 'en': 'english', 'eo': 'esperanto', 'et': 'estonian', 'tl': 'filipino', 'fi': 'finnish', 'fr': 'french', 'fy': 'frisian', 'gl': 'galician', 'ka': 'georgian', 'de': 'german', 'el': 'greek', 'gu': 'gujarati', 'ht': 'haitian creole', 'ha': 'hausa', 'haw': 'hawaiian', 'iw': 'hebrew', 'he': 'hebrew', 'hi': 'hindi', 'hmn': 'hmong', 'hu': 'hungarian', 'is': 'icelandic', 'ig': 'igbo', 'id': 'indonesian', 'ga': 'irish', 'it': 'italian', 'ja': 'japanese', 'jw': 'javanese', 'kn': 'kannada', 'kk': 'kazakh', 'km': 'khmer', 'ko': 'korean', 'ku': 'kurdish (kurmanji)', 'ky': 'kyrgyz', 'lo': 'lao', 'la': 'lat

### 步驟 4：取得翻譯的文本

In [ ]:
# Define the translation model
translator = Translator()

current_translation = translator.translate(sentence, src = "sl", dest='en')
current_translation.text

'Ljubljana is the capital of Slovenia.'

### 步驟 5：使用 Stanza 進行英文文本的語言處理


In [ ]:
!pip install -qU stanza

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 990.1/990.1 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.4/431.4 kB 21.1 MB/s eta 0:00:00


In [ ]:
import stanza

# 初始化 Stanza pipeline
stanza.download('en')
nlp_en = stanza.Pipeline('en')

# 使用 Stanza 進行處理
doc_en = nlp_en(current_translation.text)
print(doc_en)


### 步驟 6：進行詞對齊
使用 Eflomal 進行斯洛維尼亞語和英文之間的詞對齊，特別是專有名詞的對齊。

In [ ]:
# Clone the Eflomal repository
!git clone https://github.com/robertostling/eflomal

# Move into the eflomal folder
%cd /kaggle/working/eflomal

# Install Eflomal
!make
!sudo make install
!python3 setup.py install

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.4/132.4 kB 2.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# 假設已經安裝好 eflomal 並使用進行詞對齊
with open('sl.txt', 'w') as f:
    f.write(sentence)

with open('en.txt', 'w') as f:
    f.write(current_translation.text)

In [ ]:
!python3 /content/eflomal/python/scripts/eflomal-align -s "sl.txt" -t "en.txt" --model 3 -f sl-en.fwd -r sl-en.rev


In [ ]:
!cat sl-en.fwd

0-0 1-1 2-2 3-4 4-5


In [ ]:
!cat sl-en.rev

0-3 1-1 2-5 3-2


In [ ]:
print(sentence)
print(current_translation.text)

Ljubljana je glavno mesto Slovenije.
Ljubljana is the capital of Slovenia.


根據斯洛維尼亞語句子 **"Ljubljana je glavno mesto Slovenije."** 與英語句子 **"Ljubljana is the capital of Slovenia."** 的詞對齊結果，讓我們進一步解釋 `sl-en.fwd` 和 `sl-en.rev` 兩個文件的對應關係：

### 1. 斯洛維尼亞語句子：
- Ljubljana (0)
- je (1)
- glavno (2)
- mesto (3)
- Slovenije (4)

### 2. 英語句子：
- Ljubljana (0)
- is (1)
- the (2)
- capital (3)
- of (4)
- Slovenia (5)

### `sl-en.fwd` 的解釋
文件 `sl-en.fwd` 是斯洛維尼亞語到英語的對應關係：

```
0-0 1-1 2-2 3-4 4-5
```

這表示：
- 斯洛維尼亞語的第 0 個詞 **"Ljubljana"** 對應英語的第 0 個詞 **"Ljubljana"**。
- 斯洛維尼亞語的第 1 個詞 **"je"** 對應英語的第 1 個詞 **"is"**。
- 斯洛維尼亞語的第 2 個詞 **"glavno"** 對應英語的第 2 個詞 **"the"**（雖然從語意上來說，"glavno" 對應的是 "capital"，但是模型根據句法進行對齊，可能認為它對應 "the"）。
- 斯洛維尼亞語的第 3 個詞 **"mesto"** 對應英語的第 4 個詞 **"of"**（這裡可能有些偏差，因為 "mesto" 更應該對應 "capital"）。
- 斯洛維尼亞語的第 4 個詞 **"Slovenije"** 對應英語的第 5 個詞 **"Slovenia"**。

### `sl-en.rev` 的解釋
文件 `sl-en.rev` 是英語到斯洛維尼亞語的反向對應關係：

```
0-3 1-1 2-5 3-2
```

這表示：
- 英語的第 0 個詞 **"Ljubljana"** 對應斯洛維尼亞語的第 3 個詞 **"mesto"**（這個對應有些問題，"Ljubljana" 應該對應斯洛維尼亞語的第 0 個詞）。
- 英語的第 1 個詞 **"is"** 對應斯洛維尼亞語的第 1 個詞 **"je"**（這是正確的對應）。
- 英語的第 2 個詞 **"the"** 對應斯洛維尼亞語的第 5 個詞 **"Slovenije"**（這顯然不正確，"the" 應該對應 "glavno" 或被省略）。
- 英語的第 3 個詞 **"capital"** 對應斯洛維尼亞語的第 2 個詞 **"glavno"**（這是合理的對應）。

### 結論
從這兩個對應結果可以看出，對齊模型在處理專有名詞時通常能夠正確對應（如 "Ljubljana" 和 "Slovenia" 的對應），但在處理普通詞彙（如 "mesto" 對應 "of"、"the" 對應 "Slovenije"）時，可能會產生錯誤。這些錯誤可能來自詞語在句子中的語法位置不同或模型的對齊誤差。

為了糾正這些錯誤，你可以根據專有名詞（如地名、國名）的正確對應來手動修正模型的錯誤對應，或者利用 Eflomal 的雙向對齊來檢測對應是否一致，從而提高翻譯質量。